# Functor / Interpreter smoke tests

Four cases from the plan proposal:
1. `Case` / `Functor` validation
2. Algebra fold — list sum
3. Coalgebra — Lambert-style convergence (no token stream)
4. Coalgebra — Mealy machine (with token stream)
5. Coalgebra — Moore machine (same Functor, different adapter ordering)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
from streamlined import Case, Functor, CoalgResult, Interpreter
from streamlined.functor import _NO_OUTPUT

## 1. Validation

In [2]:
import traceback

# Duplicate case names → error
try:
    Functor([Case("a", 1, 0), Case("a", 0, 0)])
    print("FAIL: expected ValueError")
except ValueError as e:
    print("OK duplicate names:", e)

# Bad output value → error
try:
    Case("x", 1, 0, output=2)
    print("FAIL: expected ValueError")
except ValueError as e:
    print("OK bad output:", e)

# Unknown case name → KeyError
try:
    F = Functor([Case("step", 1, 0)])
    F["missing"]
    print("FAIL: expected KeyError")
except KeyError as e:
    print("OK unknown case:", e)

OK duplicate names: Duplicate case names in Functor
OK bad output: output must be 0 or 1
OK unknown case: "Unknown case 'missing' for this Functor"


## 2. Algebra fold — list sum

Functor: `nil` (base case) | `cons` (head :: tail).  
Algebra: sum all elements.

In [3]:
def list_decompose(lst):
    if not lst:
        return ("nil", [], [])
    return ("cons", [lst[0]], [lst[1:]])

def sum_cell(case_name, payload, child_results, params, temp):
    if case_name == "nil":
        return 0
    return payload[0] + child_results[0]

F = Functor([Case("nil", recursive=0, data=0), Case("cons", recursive=1, data=1)])
interp = Interpreter(F, sum_cell, params=None)

result = interp.run_algebra([1, 2, 3, 4, 5], list_decompose)
assert result == 15, f"Expected 15, got {result}"
print("OK algebra fold: sum([1..5]) =", result)

result_empty = interp.run_algebra([], list_decompose)
assert result_empty == 0
print("OK algebra fold: sum([]) =", result_empty)

OK algebra fold: sum([1..5]) = 15
OK algebra fold: sum([]) = 0


## 3. Coalgebra — Lambert-style convergence

No token stream; stop condition drives termination.  
Operator: scale state by 0.5 each step; converges to 0.

In [4]:
def decay_cell(state, token, params, temp):
    next_state = state * 0.5
    return CoalgResult(case_name="step", payload=[], next_states=[next_state])

prev = {}
def converged(step, state, outputs):
    if step >= 100:
        return True
    if "last" in prev and abs(state - prev["last"]) < 1e-6:
        return True
    prev["last"] = state
    return False

F = Functor([Case("step", recursive=1, data=0)])
interp = Interpreter(F, decay_cell, params=None)

outputs, final = interp.run_coalgebra(1.0, stop=converged)
assert final < 1e-5, f"Expected near 0, got {final}"
print(f"OK convergence: final state = {final:.2e} after {len(outputs)} steps (no outputs expected)")

OK convergence: final state = 9.54e-07 after 0 steps (no outputs expected)


## 4. Mealy machine — running total with per-step output

State = running sum. Output = new sum after each token.

In [5]:
def mealy_cell(state, token, params, temp):
    next_state = state + token
    return CoalgResult(
        case_name="step", payload=[token],
        next_states=[next_state], output=next_state
    )

F = Functor([Case("step", recursive=1, data=1, output=1)])
interp = Interpreter(F, mealy_cell, params=None)

outputs, final = interp.run_coalgebra(0, token_iter=[1, 2, 3, 4])
assert outputs == [1, 3, 6, 10], f"Got {outputs}"
assert final == 10
print("OK Mealy: running totals =", outputs)

OK Mealy: running totals = [1, 3, 6, 10]


## 5. Moore machine — same Functor, output before transition

Output = state *before* consuming the token (previous running total).

In [6]:
def moore_cell(state, token, params, temp):
    output = state                # emit current state first
    next_state = state + token    # then transition
    return CoalgResult(
        case_name="step", payload=[token],
        next_states=[next_state], output=output
    )

# Identical Functor declaration — interpreter can't tell Moore from Mealy
F = Functor([Case("step", recursive=1, data=1, output=1)])
interp = Interpreter(F, moore_cell, params=None)

outputs, final = interp.run_coalgebra(0, token_iter=[1, 2, 3, 4])
assert outputs == [0, 1, 3, 6], f"Got {outputs}"
assert final == 10
print("OK Moore:  lagged totals   =", outputs)

OK Moore:  lagged totals   = [0, 1, 3, 6]


## 6. Runtime validation — mismatch errors

In [7]:
# Wrong payload length from adapter
def bad_payload_cell(state, token, params, temp):
    return CoalgResult(case_name="step", payload=[token, token],  # declared data=1
                       next_states=[state])

F = Functor([Case("step", recursive=1, data=1, output=0)])
interp = Interpreter(F, bad_payload_cell, params=None)
try:
    interp.run_coalgebra(0, token_iter=[1])
    print("FAIL: expected ValueError")
except ValueError as e:
    print("OK bad payload:", e)

# Output present when case declares output=0
def unexpected_output_cell(state, token, params, temp):
    return CoalgResult(case_name="step", payload=[token],
                       next_states=[state], output=42)

F2 = Functor([Case("step", recursive=1, data=1, output=0)])
interp2 = Interpreter(F2, unexpected_output_cell, params=None)
try:
    interp2.run_coalgebra(0, token_iter=[1])
    print("FAIL: expected ValueError")
except ValueError as e:
    print("OK unexpected output:", e)

OK bad payload: Case 'step' declared data=1, adapter returned 2 payload items
OK unexpected output: Case 'step' declared output=0, but adapter output presence does not match
